In [ ]:
census_api_key = "0f03d081293f18a970ca7a84d9b2889bcdee5f65"

# County Business Patterns, 1986-2023 (bulk county files)

Downloads the Census's bulk county files (`https://www2.census.gov/programs-surveys/cbp/datasets/<year>/cbp<yy>co.zip`). These are public, so the API key above isn't used. Run the two cells below; the second one starts the download.

**Output:** `data_bulk/cbp_county_<year>.zip`, one per year (38 files, 473 MB). Each is the Census zip exactly as published, holding one comma-delimited text file. pandas reads it directly: `pd.read_csv(path, dtype=str)`. Use `dtype=str` so FIPS and industry codes keep their leading zeros.

**Things worth knowing**
- **One row per county x industry, not per county.** Nothing is collapsed. For one row per county-year, keep the all-industry total, whose industry code is `----` in the SIC years (1986-1997) and `------` after that. Every year has exactly one such row per county.
- **Columns differ by era; nothing is renamed.** The industry column is `sic` for 1986-1997 and `naics` from 1998, and the NAICS vintage changes along the way (1997, 2002, 2007, 2012, 2017). Other changes: the `*_nf` flag columns start in 2007; `empflag` is gone from 2018; the size class `n1_4` becomes `n<5` in 2017; `censtate`/`cencty` (Census's own codes) are gone from 2020. The 2015 file has upper-case column names, so lower-case the header when stacking years.
- **County code `999` is a statewide row.** Every year's file has them (in 2023, one set for each of the 50 states and none for DC), so drop `fipscty == "999"` when you want counties only. `cbp_census_pull.ipynb` notes that the API instead puts the 2023 statewide rows in the state file, so the two pulls won't match on this.
- **No padding rows.** Unlike the API, these files don't pad with empty cells; only 35 rows in 1986-1989 have `est == 0`, and none do after that.
- **Duplicates in 1999.** Six rows repeat a (state, county, NAICS) key in Alabama county 045: five are exact copies and one pair (`62111/`) disagrees. Nothing else is duplicated in any year.
- **Totals look right.** Summing the all-industry rows over counties gives 5.8M establishments and 83M employees in 1986, rising to 8.4M and 140M in 2023. These are sums of published county cells, so they can differ slightly from the Census's national figures (suppression, and noise infusion from 2007).
- **2024 isn't out yet.** `cbp24co.zip` returns 404 (checked 2026-10-31), so `LAST_YEAR` is 2023.
- **Resumable.** A file gets its final name only after it has downloaded in full, matched the server's `Content-Length`, and passed a zip CRC check. Re-running skips finished years and lists any failures at the end.

In [ ]:
import time
import zipfile
from pathlib import Path

import requests

URL = "https://www2.census.gov/programs-surveys/cbp/datasets/{year}/cbp{yy:02d}co.zip"
FIRST_YEAR, LAST_YEAR = 1986, 2023  # the Census has not published 2024 yet
CHUNK = 1 << 20
RETRY_STATUS = {429, 500, 502, 503, 504}


def zip_problem(path):
    """What is wrong with the zip at `path`, or None if it is intact."""
    if not zipfile.is_zipfile(path):
        return "not a zip file"
    with zipfile.ZipFile(path) as z:
        bad = z.testzip()  # re-reads every member and checks its CRC
    return f"corrupt member {bad}" if bad else None


def download_year(session, year, path, tries=5):
    """Download one year's county zip to `path` via a .part file; returns its size in bytes."""
    url = URL.format(year=year, yy=year % 100)
    part = path.with_name(path.name + ".part")
    for attempt in range(tries):
        try:
            with session.get(url, stream=True, timeout=120) as resp:
                if resp.status_code == 200:
                    with open(part, "wb") as f:
                        for chunk in resp.iter_content(CHUNK):
                            f.write(chunk)
                    expected = int(resp.headers.get("Content-Length", -1))
                    size = part.stat().st_size
                    problem = (
                        f"truncated: got {size:,} of {expected:,} bytes"
                        if expected >= 0 and size != expected
                        else zip_problem(part)
                    )
                    if problem is None:
                        part.replace(path)
                        return size
                elif resp.status_code in RETRY_STATUS:
                    problem = f"HTTP {resp.status_code}"
                else:
                    raise RuntimeError(f"HTTP {resp.status_code} from {url}")
        except requests.RequestException as e:
            problem = str(e)
        if attempt < tries - 1:
            time.sleep(min(2**attempt, 30))
    part.unlink(missing_ok=True)
    raise RuntimeError(f"gave up on {url} after {tries} tries: {problem}")


def pull_cbp_bulk(years=range(FIRST_YEAR, LAST_YEAR + 1), out_dir="data_bulk", overwrite=False):
    """Download the bulk CBP county file for each year into `out_dir`/cbp_county_<year>.zip.

    A file only gets its final name once it has downloaded in full and passed the zip check,
    so finished files are skipped on re-runs unless `overwrite`. Returns a list of (year, error)
    failures.
    """
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    failures = []
    for year in years:
        path = Path(out_dir) / f"cbp_county_{year}.zip"
        if path.exists() and not overwrite:
            print(f"{year}: {path.name} exists, skipping", flush=True)
            continue
        start = time.time()
        try:
            size = download_year(session, year, path)
        except Exception as e:
            print(f"{year}: FAILED: {e}", flush=True)
            failures.append((year, str(e)))
            continue
        print(f"{year}: {size / 1e6:.1f} MB -> {path} ({time.time() - start:.0f}s)", flush=True)
    if failures:
        print(f"\n{len(failures)} failed (re-run to retry them):", flush=True)
        for year, err in failures:
            print(f"  {year}: {err}", flush=True)
    return failures

In [ ]:
# Every county file, 1986-2023 -> data_bulk/cbp_county_<year>.zip (about 500 MB, a minute or two).
# Finished years are skipped if you re-run it.
failures = pull_cbp_bulk()